# Análisis de Eficiencia Espacial: Detour Factor (Factor de Desviación)

El **Detour Factor** (o coeficiente de circuidad) mide la eficiencia de una red de transporte comparando la distancia real recorrida a través de la infraestructura contra la distancia teórica en línea recta (Haversine).

**Fórmula:**
$$DF = \frac{Distancia_{Red} + Distancia_{Caminata}}{Distancia_{Haversine}}$$

Un valor de **1.0** indica una ruta perfectamente directa. Valores superiores a **1.5** suelen indicar ineficiencias topológicas, barreras urbanas o rodeos excesivos por transbordos.

En este notebook realizaremos:
1. Análisis estadístico de 500 viajes aleatorios.
2. Visualización de la trazabilidad y sistemas involucrados.
3. Mapeo interactivo de rutas específicas.

In [15]:
# ==========================================
# 2. Inicialización del Entorno y Librerías
# ==========================================
%load_ext autoreload
%autoreload 2
%matplotlib inline 

import sys
import os
import folium
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
### Visualizador de utils
from src.core.utils.utils_detaur_factor import render_vft_detour_map

# Silenciar advertencias futuras para un reporte limpio
warnings.filterwarnings('ignore', category=FutureWarning)

# Asegurar que Jupyter encuentra la carpeta raíz del proyecto
proyecto_path = os.path.abspath('..')
if proyecto_path not in sys.path:
    sys.path.append(proyecto_path)

from src.infrastructure.go_client.client import fetch_full_network
from src.api.schemas.schemas import GeoJSONTransportSchema
from src.core.services.graph_builder import VFTGraphBuilder
from src.core.algorithms.topologicalIndicators.detaurFactor import DetourFactorOrchestrator
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'sans-serif'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
# ==========================================
# 3. Descarga de Red y Construcción del Grafo
# ==========================================
print("📥 Descargando red de transporte...")
raw_data = await fetch_full_network()

print("⚙️ Construyendo Grafo Dirigido VFT...")
validated_payload = GeoJSONTransportSchema(**raw_data)

plt.ioff() 
GRAF= VFTGraphBuilder(validated_payload)
G = GRAF.build_graph()
plt.close('all')
plt.ion()

orchestrator = DetourFactorOrchestrator(G)
print(f"✅ Grafo construido: {G.number_of_nodes()} nodos topológicos.")

📥 Descargando red de transporte...
2026-04-26 20:19:37 | INFO     | VFT_Model | Construyendo el puente hacia el módulo espacial de Go...
2026-04-26 20:19:37 | INFO     | VFT_Model | Solicitando capa espacial a: http://localhost:8080/movilidad/mapas/geojsonEstacion
2026-04-26 20:19:37 | INFO     | VFT_Model | Solicitando capa espacial a: http://localhost:8080/movilidad/mapas/geojsonLinea
2026-04-26 20:19:38 | INFO     | VFT_Model | Red extraída: 22766 entidades espaciales listas para VFT.
⚙️ Construyendo Grafo Dirigido VFT...
2026-04-26 20:19:38 | INFO     | VFT_Model | Iniciando VFTGraphBuilder en modo: REALISTIC_INTEGRATION
2026-04-26 20:19:38 | INFO     | VFT_Model | Fase 1 y 2: Extrayendo nodos y trazos base...
2026-04-26 20:19:41 | INFO     | VFT_Model | Fase 2 completada: 64459 aristas interestación creadas.
2026-04-26 20:19:41 | INFO     | VFT_Model | Grafo Base construido: 11115 Nodos y 32103 Segmentos.
2026-04-26 20:19:41 | WARNING  | VFT_Model | Eliminadas 1543 aristas phantom

## 1. Muestreo Estadístico y Caso de Prueba Inicial
En esta sección generamos una muestra aleatoria de 100 viajes para entender el comportamiento base de la red y visualizamos el primer resultado obtenido sin ningún criterio de ordenamiento.

In [22]:
# 1. Generar la muestra estadística global
tamano_muestra = 100
print(f"🎲 Generando muestra de {tamano_muestra} rutas...")
muestra_json = orchestrator.calculate_sample_routes(sample_size=tamano_muestra, seed=42, return_json=True)

if muestra_json:
    # Crear DataFrame de métricas
    df_metrics = pd.DataFrame([m['metrics'] for m in muestra_json])
    
    # Resumen estadístico
    print("\n📊 ESTADÍSTICAS GLOBALES DEL DETOUR FACTOR:")
    display(df_metrics['Factor_Desviacion'].describe().to_frame().T)
    
    # Visualizar el primer caso de la lista
    primer_caso = muestra_json[0]
    mapa_ini, _, _, _, _, _ = render_vft_detour_map(
        primer_caso, 
        title=f"CASO INICIAL ALEATORIO (DF: {primer_caso['metrics']['Factor_Desviacion']})"
    )
    display(mapa_ini)

🎲 Generando muestra de 100 rutas...

📊 ESTADÍSTICAS GLOBALES DEL DETOUR FACTOR:


,count,mean,std,min,25%,50%,75%,max
Factor_Desviacion,100.0,1.6075,0.544622,1.0,1.25,1.42,1.78,4.27


🗺️ CASO INICIAL ALEATORIO (DF: 1.08)
📍 Punto cerca de Tepehuanos La Joya ➡️ Punto cerca de Acoxpa
📊 DF: 1.08 | Red: 4.2km vs Teórica: 3.87km


## 2. Análisis de Máxima Eficiencia: Las 5 Mejores Rutas
Aquí identificamos los trayectos donde la red de transporte es más competitiva frente al automóvil (trayectos casi directos). Visualizamos el mapa de la ruta con el menor Factor de Desviación.

In [23]:
# 1. Mostrar tabla de las 5 mejores
print("✅ TOP 5 RUTAS MÁS EFICIENTES (Cercanas a 1.0):")
df_mejores = df_metrics.sort_values(by="Factor_Desviacion", ascending=True).head(5)
display(df_mejores[['Origen', 'Destino', 'Sistemas_Involucrados', 'Distancia_Red_km', 'Factor_Desviacion']])

# 2. Identificar y graficar el mejor caso absoluto
mejor_caso = min(muestra_json, key=lambda x: x['metrics']['Factor_Desviacion'])
mapa_mejor, _, _, _, _, _ = render_vft_detour_map(
    mejor_caso, 
    title=f"LA RUTA MÁS EFICIENTE (DF: {mejor_caso['metrics']['Factor_Desviacion']})"
)
display(mapa_mejor)

✅ TOP 5 RUTAS MÁS EFICIENTES (Cercanas a 1.0):


,Origen,Destino,Sistemas_Involucrados,Distancia_Red_km,Factor_Desviacion
82,Punto cerca de Plutarco Elias Calles,Punto cerca de Fuente de Loreto - Av. Guelatao,[RTP],1.02,1.00
79,Punto cerca de Paseo de la Reforma - Fuente de...,Punto cerca de El Ángel,"[CC, TROLE]",3.56,1.03
0,Punto cerca de Tepehuanos La Joya,Punto cerca de Acoxpa,[RTP],4.20,1.08
30,Punto cerca de Antiguo Camino a Xochimilco - D...,Punto cerca de Emiliano Zapata - Cuauhtémoc,[CC],1.58,1.08
13,Punto cerca de C. C. Arenal Cuarta Sección - X...,Punto cerca de Iztacihuatl - Cacamatzin,[RTP],0.78,1.08


🗺️ LA RUTA MÁS EFICIENTE (DF: 1.0)
📍 Punto cerca de Plutarco Elias Calles ➡️ Punto cerca de Fuente de Loreto - Av. Guelatao
📊 DF: 1.0 | Red: 1.02km vs Teórica: 1.02km


## 3. Análisis de Puntos Críticos: Las 5 Peores Rutas
Identificamos los trayectos que presentan mayores ineficiencias, ya sea por falta de conexiones directas o rodeos excesivos. Mapeamos el "Peor Caso" para analizar visualmente la falla en la topología.

In [24]:
# 1. Mostrar tabla de las 5 peores
print("🚨 TOP 5 RUTAS CON MAYOR INEFICIENCIA (Rodeos Críticos):")
df_peores = df_metrics.sort_values(by="Factor_Desviacion", ascending=False).head(5)
display(df_peores[['Origen', 'Destino', 'Sistemas_Involucrados', 'Distancia_Red_km', 'Factor_Desviacion']])

# 2. Identificar y graficar el peor caso absoluto
peor_caso = max(muestra_json, key=lambda x: x['metrics']['Factor_Desviacion'])
mapa_peor, _, _, _, _, _ = render_vft_detour_map(
    peor_caso, 
    title=f"LA RUTA MÁS INEFICIENTE (DF: {peor_caso['metrics']['Factor_Desviacion']})"
)
display(mapa_peor)

🚨 TOP 5 RUTAS CON MAYOR INEFICIENCIA (Rodeos Críticos):


,Origen,Destino,Sistemas_Involucrados,Distancia_Red_km,Factor_Desviacion
37,Punto cerca de Carlos A. Vidal - Everardo Gonz...,Punto cerca de 20 de Noviembre - Gargolas,"[RTP, METRO, TL, CC]",48.96,4.27
57,Punto cerca de Las Fuentes,Punto cerca de Cuauhtémoc,[MEXIBÚS],1.32,3.61
71,Punto cerca de Terminal Aérea,Punto cerca de Pekin y Japón,"[TROLE, CC]",3.07,2.90
48,Punto cerca de Av. Chapultepec - Napoles,Punto cerca de Marina Nacional - Carrillo Puerto,"[RTP, METRO, CC]",11.93,2.87
2,Punto cerca de Circuito Interior - Eje Central...,Punto cerca de Prol. Paseo de la Reforma - Gui...,"[RTP, CC, METRO, TL]",46.90,2.79


🗺️ LA RUTA MÁS INEFICIENTE (DF: 4.27)
📍 Punto cerca de Carlos A. Vidal - Everardo González ➡️ Punto cerca de 20 de Noviembre - Gargolas
📊 DF: 4.27 | Red: 48.96km vs Teórica: 11.46km


## 4. Análisis de Ruta Arbitraria Personalizada (Caso de Estudio)
En esta sección evaluamos un viaje específico utilizando coordenadas exactas (Longitud, Latitud). Esto permite analizar la "primera y última milla" (caminata hacia la estación más cercana) y el factor de desviación para trayectos reales.

**Caso de Estudio:**
* **Origen:** Zona Sur, fuera del Anillo Periférico.
* **Destino:** Zona Oriente, inmediaciones del Autódromo Hermanos Rodríguez.

## 5. Galería de Casos de Estudio: Análisis de Trayectos Específicos
En esta sección, evaluamos una serie de rutas estratégicas seleccionadas por su relevancia en la conectividad de la Ciudad de México. 

Cada caso analiza:
1. **Contexto Geográfico:** Por qué son importantes estos puntos.
2. **Eficiencia Topológica:** Cómo se comporta la red frente a la distancia directa.
3. **Métrica de Caminata:** La distancia necesaria para conectar con la red formal (Primera y Última Milla).

In [41]:
from IPython.display import display, Markdown

# 1. DEFINICIÓN DE RUTAS (Agrega aquí tantas como necesites)
casos_estudio = [
    {
        "titulo": "Caso 1: Periferia Extrema Sur hacia Centro-Oriente",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico hacia el Hub de entretenimiento del Autódromo. Evalúa la dificultad de salir de zonas con baja densidad de transporte formal.",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.09342206113304, 19.4059272938029)
    },
    {
        "titulo": "Caso 2: Conectividad Sur-Centro/Poniente (Milpa Alta a Reforma)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Sur) hacia el CBD Reforma.",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.17161535569772, 19.42527244485531) 
    },
    {
        "titulo": "Caso 3: Conectividad Sur-Poniente (Milpa Alta a Santa Fé)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Sur) hacia el CBD Santa Fé.",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.26937963016258 ,19.357049109027344) 
    },
    {
        "titulo": "Caso 4: Conectividad Sur-Poniente Norte (Milpa Alta al punto más cercano a FES Acatlán)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Sur) hacia Centro Educativo FES Acatlán.",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.2098132566566, 19.489900115538866) 
    },
    {
        "titulo": "Caso 5: Conectividad Sur-Centro/Sur (Milpa Alta a CU)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Sur) hacia el Centro Educativo Ciudad Universitaria.",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.184797668633, 19.33163949142085)
    },
    {
        "titulo": "Caso 6: Conectividad Sur-Centro/Sur (Milpa Alta a Coyoacán)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Sur) hacia Zona Residencial Centro Sur coyoacán.",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.15633349395276, 19.342445017191288)
    },
    {
        "titulo": "Caso 7: Conectividad Oriente-Centro/Sur (Santa Catarina a CU)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Oriente) hacia Centro Educativo Ciudad Universitaria.",
        "coords_o": (-98.96533517329291, 19.315314989332904),
        "coords_d": (-99.184797668633, 19.33163949142085) 
    },
    {
        "titulo": "Caso 8: Conectividad Oriente-Centro/Sur (Santa Catarina al Punto más cercano a FES Acatán)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Oriente) hacia Centro Educativo FES Acatlán.",
        "coords_o": (-98.96533517329291, 19.315314989332904),
        "coords_d": (-99.2098132566566, 19.489900115538866) 
    },
    {
        "titulo": "Caso 9: Conectividad Sur-Oriente (Milpa Alta - Tláhuac/Santa Catarina)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Sur) hacia zonas residenciales fuera del Anillo Periférico (Oriente).",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-98.9648158208182, 19.315197623324714) 
    }
]

# 2. BUCLE OPTIMIZADO DE RENDERIZADO
for caso in casos_estudio:
    # Presentación elegante del título y descripción
    display(Markdown(f"### {caso['titulo']}"))
    display(Markdown(f"**Contexto:** {caso['descripcion']}"))
    
    # Cálculo
    res = orchestrator.calculate_custom_route(caso['coords_o'], caso['coords_d'], return_json=True)
    
    if res:
        # Renderizado del mapa y obtención de métricas robustas
        mapa, ini, fin, df_val, d_hav, d_red = render_vft_detour_map(
            res, 
            title=f"Visualización: {caso['titulo']}"
        )
        
        # Resumen de métricas en formato Markdown para mejor lectura
        metricas_caminata = res['metrics'].get('Consideraciones_Reales', {})
        c1 = metricas_caminata.get('Caminata_1ra_Milla_Metros', 0)
        c2 = metricas_caminata.get('Caminata_Ultima_Milla_Metros', 0)
        
        resumen_md = f"""
| Métrica | Valor |
| :--- | :--- |
| **Punto de Abordaje** | {metricas_caminata.get('Estacion_Abordaje', 'N/A')} |
| **Punto de Descenso** | {metricas_caminata.get('Estacion_Descenso', 'N/A')} |
| **Factor de Desviación** | `{df_val}` |
| **Distancia en Red** | {d_red} km |
| **Distancia en Haversine** | {d_hav} km |
| **Caminata Total (1ra + Última)** | {round(c1 + c2, 2)} metros |
"""
        display(Markdown(resumen_md))
        display(mapa)
        display(Markdown("---")) # Separador visual entre mapas
    else:
        display(Markdown(f"⚠️ *No se pudo calcular la ruta para: {caso['titulo']}*"))

### Caso 1: Periferia Extrema Sur hacia Centro-Oriente

**Contexto:** Trayecto desde zonas residenciales fuera del Anillo Periférico hacia el Hub de entretenimiento del Autódromo. Evalúa la dificultad de salir de zonas con baja densidad de transporte formal.

🗺️ Visualización: Caso 1: Periferia Extrema Sur hacia Centro-Oriente
📍 Punto cerca de Niños Héroes y San Gregorio - Oaxtepec ➡️ Punto cerca de Viaducto y Puerta 5
📊 DF: 1.52 | Red: 34.35km vs Teórica: 22.59km



| Métrica | Valor |
| :--- | :--- |
| **Punto de Abordaje** | Niños Héroes y San Gregorio - Oaxtepec |
| **Punto de Descenso** | Viaducto y Puerta 5 |
| **Factor de Desviación** | `1.52` |
| **Distancia en Red** | 34.35 km |
| **Distancia en Haversine** | 22.59 km |
| **Caminata Total (1ra + Última)** | 0 metros |


---

...

### Caso 9: Conectividad Sur-Oriente (Milpa Alta - Tláhuac/Santa Catarina)

**Contexto:** Trayecto desde zonas residenciales fuera del Anillo Periférico (Sur) hacia zonas residenciales fuera del Anillo Periférico (Oriente).

🗺️ Visualización: Caso 9: Conectividad Sur-Oriente (Milpa Alta - Tláhuac/Santa Catarina)
📍 Punto cerca de Niños Héroes y San Gregorio - Oaxtepec ➡️ Punto cerca de Eje 10 Sur - Fábrica de Tubos
📊 DF: 2.52 | Red: 37.28km vs Teórica: 14.79km



| Métrica | Valor |
| :--- | :--- |
| **Punto de Abordaje** | Niños Héroes y San Gregorio - Oaxtepec |
| **Punto de Descenso** | Eje 10 Sur - Fábrica de Tubos |
| **Factor de Desviación** | `2.52` |
| **Distancia en Red** | 37.28 km |
| **Distancia en Haversine** | 14.79 km |
| **Caminata Total (1ra + Última)** | 0 metros |


---